In [3]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction import DictVectorizer
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import root_mean_squared_error

url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"
df = pd.read_csv(url)
df.describe()

df = df.fillna(0)

df_full_train, df_test = train_test_split(
    df, test_size=0.2, random_state=1
)

df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

y_train = df_train.fuel_efficiency_mpg.values
y_val = df_val.fuel_efficiency_mpg.values

del df_train["fuel_efficiency_mpg"]
del df_val["fuel_efficiency_mpg"]

dv = DictVectorizer(sparse=True)

train_dicts = df_train.to_dict(orient="records")
val_dicts = df_val.to_dict(orient="records")

X_train = dv.fit_transform(train_dicts)
X_val = dv.transform(val_dicts)

```
1. Original splitting feature (1 point)


vehicle_weight

model_year

origin

fuel_type
```

In [4]:
dt = DecisionTreeRegressor(max_depth=1)
dt.fit(X_train, y_train)

feature = dv.feature_names_[dt.tree_.feature[0]]
print(feature)

model_year


```
2. RMSE on validation (1 point)


0.1837

1.837

18.37

183.7
```

In [5]:
rf = RandomForestRegressor(
    n_estimators=10,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

y_pred = rf.predict(X_val)

rmse = root_mean_squared_error(y_val, y_pred)
print(rmse)

1.8332197358745623


```
3. Number of estimators (1 point)


10

50

100

150
```

In [6]:
for n in [10, 50, 100, 150]:
    rf = RandomForestRegressor(
        n_estimators=n,
        random_state=1,
        n_jobs=-1
    )

    rf.fit(X_train, y_train)
    pred = rf.predict(X_val)

    rmse = root_mean_squared_error(y_val, pred)
    print(n, round(rmse, 3))

10 1.833
50 1.774
100 1.77
150 1.769


```
4. Best max depth (1 point)


10

15

20

25
```

In [7]:
for depth in [10, 15, 20, 25]:
    scores = []

    for n in [10, 50, 100, 150]:
        rf = RandomForestRegressor(
            n_estimators=n,
            max_depth=depth,
            random_state=1,
            n_jobs=-1
        )

        rf.fit(X_train, y_train)
        pred = rf.predict(X_val)

        rmse = root_mean_squared_error(y_val, pred)
        scores.append(rmse)

    print(depth, np.mean(scores))

10 1.7543924943801008
15 1.7842243445385957
20 1.7867711878395158
25 1.7863054936840876


```
5. Most important feature (1 point)


vehicle_weight

horsepower

acceleration

engine_displacement

```

In [8]:
rf = RandomForestRegressor(
    n_estimators=10,
    max_depth=20,
    random_state=1,
    n_jobs=-1
)

rf.fit(X_train, y_train)

importance = pd.Series(
    rf.feature_importances_,
    index=dv.feature_names_
)

print(importance.sort_values(ascending=False))

model_year                      0.296107
vehicle_weight                  0.203313
fuel_type=Gasoline              0.160712
horsepower                      0.078271
engine_displacement             0.061281
acceleration                    0.054998
origin=USA                      0.042152
drivetrain=Front-wheel drive    0.020062
num_doors                       0.019401
fuel_type=Diesel                0.016571
fuel_type=Hybrid                0.016272
drivetrain=All-wheel drive      0.007951
num_cylinders                   0.007318
origin=Asia                     0.006515
drivetrain=Rear-wheel drive     0.005638
origin=Europe                   0.003436
dtype: float64


```
6. XGBoost eta (1 point)


0.3

0.1

Both give equal value
```

In [9]:
import sys
!{sys.executable} -m ensurepip --upgrade

Looking in links: c:\Users\Admin3\AppData\Local\Temp\tmp6eibdjm8


In [10]:
import sys
!{sys.executable} -m pip install --upgrade pip
!{sys.executable} -m pip install xgboost

In [12]:
import xgboost as xgb

features = dv.get_feature_names_out().tolist()

dtrain = xgb.DMatrix(
    X_train,
    label=y_train,
    feature_names=features
)

dval = xgb.DMatrix(
    X_val,
    label=y_val,
    feature_names=features
)

watchlist = [(dtrain, "train"), (dval, "val")]

for eta in [0.3, 0.1]:

    xgb_params = {
        "eta": eta,
        "max_depth": 6,
        "min_child_weight": 1,
        "objective": "reg:squarederror",
        "nthread": 8,
        "seed": 1,
        "verbosity": 1,
    }

    model = xgb.train(
        xgb_params,
        dtrain,
        num_boost_round=100,
        evals=watchlist,
        verbose_eval=False
    )

    pred = model.predict(dval)

    rmse = root_mean_squared_error(y_val, pred)

    print(f"eta={eta}, RMSE={rmse:.4f}")

eta=0.3, RMSE=1.8265
eta=0.1, RMSE=1.7248
